# 04 — SFT (Arm A)

Supervised fine-tuning of **Qwen3-0.6B-Base** on `sft_all` (122,416 MATH-style examples) with completion-only loss.
The resulting model is **Arm A** and the starting point for every later arm (B: +DPO, C: +GRPO, D: +DPO→GRPO).

**Two modes** (set in the config cell):
- `SMOKE_TEST = True` — 50 steps on 2,000 examples. Checks the pipeline end to end (data, loss mask, memory, W&B). No Hub push.
- `SMOKE_TEST = False` — the full 2-epoch run. Pushes resumable checkpoints to the Hub at every save.

**Long runs: launch in the background, not with Run All** (the SSH connection to the Space drops; a background job doesn't care):
```bash
cd ~/app/DPO_King_Research
nohup jupyter nbconvert --to notebook --execute 04_sft.ipynb \
  --output 04_sft_run.ipynb --ExecutePreprocessor.timeout=-1 > sft_run.log 2>&1 &
tail -f sft_run.log      # Ctrl+C stops watching, not training
```

**After a crash / Space restart:** redo the terminal setup, set `RESUME = True`, set `PRIOR_GPU_HOURS` to the hours already spent, save (Cmd+S), and relaunch with the same command. It resumes from `last-checkpoint/` on the Hub (weights + optimizer + scheduler + RNG state).

In [ ]:
# ---- Environment (must be set before torch / wandb are imported) ----
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less memory fragmentation
os.environ["WANDB_PROJECT"] = "dpo-king"

import json
import time

import torch
import transformers
import trl
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

from posttrain.prompts import to_prompt_completion, IM_END

print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"trl {trl.__version__} | datasets {datasets.__version__}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ---- Config ----
SMOKE_TEST = False       # True = 50-step pipeline check, False = full run
RESUME = False           # True only when relaunching after a crash (needs last-checkpoint/ on the Hub)
PRIOR_GPU_HOURS = 0.0    # when resuming: GPU-hours already spent on this run before the crash

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
HUB_ID = "Maci10/dpo-king-sft"
TRAIN_REPO = "Maci10/dpo-king-sft_all"
VAL_REPO = "Maci10/dpo-king-math_val"
OUTPUT_DIR = "checkpoints/sft"
SEED = 42

LEARNING_RATE = 1e-5
NUM_EPOCHS = 2
MAX_LENGTH = 1024        # p95 of the data is 445 tokens; longer examples were already dropped in 01
WARMUP_RATIO = 0.03      # transformers v5: pass as warmup_steps (a float < 1 means "fraction of steps")
LR_SCHEDULER = "cosine"
WEIGHT_DECAY = 0.0
USE_WANDB = True

# Batch: effective batch = PER_DEVICE_BS * GRAD_ACCUM = 64 in both modes
if SMOKE_TEST:
    PER_DEVICE_BS, GRAD_ACCUM = 4, 16     # fits a 24 GB A10G
else:
    PER_DEVICE_BS, GRAD_ACCUM = 16, 4     # A100 80 GB

RUN_NAME = "armA_sft_smoke" if SMOKE_TEST else "armA_sft_v3"

# Fixed W&B run ID: a resumed run continues the same W&B charts instead of starting a new run
os.environ["WANDB_RUN_ID"] = RUN_NAME.replace("_", "-")
os.environ["WANDB_RESUME"] = "allow"

print(f"{RUN_NAME} | smoke={SMOKE_TEST} | resume={RESUME} | "
      f"batch {PER_DEVICE_BS} x {GRAD_ACCUM} = {PER_DEVICE_BS * GRAD_ACCUM}")

In [ ]:
# ---- Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Base model: EOS = PAD = <|endoftext|>. The completion ends with <|im_end|>, which is what teaches the model to stop.
print("eos:", tokenizer.eos_token, "| pad:", tokenizer.pad_token)
print("<|im_end|> id:", tokenizer.convert_tokens_to_ids(IM_END))

In [ ]:
# ---- Data ----
def load_single_split(repo):
    # Each dataset repo has exactly one split ("train" or "test"); take whichever it is.
    dd = load_dataset(repo)
    assert len(dd) == 1, f"{repo} has splits {list(dd)}"
    return next(iter(dd.values()))

def to_sft_format(ds):
    # SFTTrainer wants exactly two text columns: prompt and completion
    if "prompt" not in ds.column_names:
        ds = ds.map(to_prompt_completion)
    return ds.select_columns(["prompt", "completion"])

train_ds = to_sft_format(load_single_split(TRAIN_REPO))
val_ds = to_sft_format(load_single_split(VAL_REPO))

if SMOKE_TEST:
    train_ds = train_ds.shuffle(seed=SEED).select(range(2000))
    val_ds = val_ds.shuffle(seed=SEED).select(range(100))

print(f"train: {len(train_ds):,} | val: {len(val_ds):,}")
print("\n--- PROMPT ---\n" + train_ds[0]["prompt"])
print("--- COMPLETION ---\n" + train_ds[0]["completion"])

In [ ]:
# ---- Trainer ----
steps_per_epoch = len(train_ds) // (PER_DEVICE_BS * GRAD_ACCUM)
save_every = 50 if SMOKE_TEST else max(steps_per_epoch // 4, 50)   # 4 evals/saves per epoch
print(f"steps/epoch: {steps_per_epoch} | eval+save every {save_every} steps")

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    seed=SEED,

    # data / loss
    max_length=MAX_LENGTH,
    completion_only_loss=True,     # loss only on the solution, prompt tokens get label -100
    eos_token=IM_END,
    packing=False,

    # optimization
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    max_steps=50 if SMOKE_TEST else -1,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    # memory / precision
    bf16=True,
    gradient_checkpointing=True,
    model_init_kwargs={"dtype": torch.bfloat16},

    # logging / eval / saving
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=save_every,
    save_strategy="steps",
    save_steps=save_every,
    save_total_limit=2,            # only on the Space disk; every save is still a commit on the Hub
    report_to="wandb" if USE_WANDB else "none",

    # Hub: "checkpoint" also pushes last-checkpoint/ (optimizer etc.) so a crashed run can resume
    push_to_hub=not SMOKE_TEST,
    hub_model_id=HUB_ID,
    hub_private_repo=True,
    hub_strategy="checkpoint",
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

In [ ]:
# ---- Verify the loss mask (TRL 1.18 stores it in `labels`: -100 = ignored) ----
ex = trainer.train_dataset[0]
ids, labels = ex["input_ids"], ex["labels"]
trained = [t for t, l in zip(ids, labels) if l != -100]
ignored = [t for t, l in zip(ids, labels) if l == -100]

print(f"total {len(ids)} | trained {len(trained)} | ignored (prompt) {len(ignored)}")
print("\n--- IGNORED (should be the prompt) ---\n" + tokenizer.decode(ignored))
print("\n--- TRAINED (should be the solution ending in <|im_end|>) ---\n" + tokenizer.decode(trained))

im_end_id = tokenizer.convert_tokens_to_ids(IM_END)
assert trained[-1] == im_end_id, "completion should end with <|im_end|>"
assert trained.count(im_end_id) == 1, "exactly one <|im_end|> should be trained"
print("\nLoss mask OK")

In [ ]:
# ---- Train (fresh or resumed) ----
t0 = time.time()

if RESUME:
    from huggingface_hub import snapshot_download
    snapshot_download(HUB_ID, allow_patterns="last-checkpoint/*", local_dir=OUTPUT_DIR)
    train_result = trainer.train(resume_from_checkpoint=f"{OUTPUT_DIR}/last-checkpoint")
else:
    train_result = trainer.train()

train_secs = time.time() - t0
gpu_hours = train_secs / 3600
total_gpu_hours = PRIOR_GPU_HOURS + gpu_hours
print(f"Training time: {train_secs/60:.1f} min = {gpu_hours:.3f} GPU-hours this session "
      f"({total_gpu_hours:.3f} total) on {torch.cuda.get_device_name(0)}")
print(train_result.metrics)

In [ ]:
# ---- Save, summarize, push ----
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

final_eval = trainer.evaluate()
summary = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "hub_id": HUB_ID if not SMOKE_TEST else None,
    "smoke_test": SMOKE_TEST,
    "resumed": RESUME,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_hours_this_session": round(gpu_hours, 4),
    "gpu_hours_total": round(total_gpu_hours, 4),
    "n_train": len(train_ds),
    "n_val": len(val_ds),
    "config": {
        "learning_rate": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "per_device_bs": PER_DEVICE_BS, "grad_accum": GRAD_ACCUM,
        "effective_batch": PER_DEVICE_BS * GRAD_ACCUM, "max_length": MAX_LENGTH,
        "warmup_ratio": WARMUP_RATIO, "scheduler": LR_SCHEDULER, "seed": SEED,
    },
    "train_metrics": train_result.metrics,
    "final_eval": final_eval,
}
with open(f"{final_dir}/sft_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

if not SMOKE_TEST:
    trainer.push_to_hub(commit_message=f"End of training: {RUN_NAME}")
    from huggingface_hub import HfApi
    HfApi().upload_file(path_or_fileobj=f"{final_dir}/sft_summary.json",
                        path_in_repo="sft_summary.json", repo_id=HUB_ID)
    print(f"Pushed to https://huggingface.co/{HUB_ID}")

## Results log

### Smoke test — `armA_sft_smoke` (A10G 24 GB, Nov 17)
- 50 steps on 2,000 examples, batch 4 × 16 = 64, gradient checkpointing on
- 3:23 wall clock (~4 s/step)
- train loss 0.349 | val loss 0.830 | mean token accuracy 0.783 | entropy 0.628
- Loss mask verified: prompt ignored, solution trained, exactly one `<|im_end|>`
- Lessons: `warmup_ratio` removed in transformers v5 → `warmup_steps=0.03`; OOM at batch 8 without checkpointing on the A10G

### Full run — `armA_sft_v3` (A100-SXM4 80 GB, Nov 18)
- 122,416 examples, 2 epochs = 3,826 steps, batch 16 × 4 = 64, ~0.65 steps/s
- Crashed after step 1912 (Space froze); resumed from `last-checkpoint/` and finished
- GPU-hours: ~0.82 (steps 0–1912) + 0.826 (steps 1912–3826) = **~1.65 A100-hours**

| Step | Epoch | Train loss | Val loss | Mean token acc |
|---|---|---|---|---|
| 2390 | 1.25 | 0.2571 | 0.7150 | 0.8044 |
| 2868 | 1.50 | 0.2635 | 0.7152 | 0.8042 |
| 3346 | 1.75 | 0.2676 | 0.7143 | 0.8043 |
| 3826 | 2.00 | 0.2545 | 0.7144 | 0.8045 |

Observations:
- Val loss is flat through epoch 2 (≈0.714 from step ~1434 on). The second epoch barely moved it.
- Large train/val gap (0.25 vs 0.71). Likely cause: the train set has many MetaMathQA rewrites of the same problems in one consistent style, while val is original MATH solutions in a different style. Loss ≠ accuracy, so the real test is graded accuracy.

### Failed attempts (count toward total compute)
- `armA_sft` (v1): froze at step 2212 (~0.94 h). No optimizer state on the Hub (`hub_strategy="every_save"`), so it could not resume. Weights kept at steps 478 / 956 / 1434 / 1912.
- `armA_sft_v2`: died ~17 min in (~0.3 h).
- Fix: `hub_strategy="checkpoint"` + `save_total_limit=2` + background launch with `nohup`.

## Next steps
1. In `03_eval`, grade the saved checkpoints on **math_val** (accuracy, not loss) and pick the best one.
2. Run **MATH-500 once** on the chosen checkpoint as Arm A (`ARM="armA_sft"`, zero-shot only).
3. Optional for the paper: accuracy vs step plot across checkpoints, next to the flat val-loss curve.
4. Load a specific checkpoint by its weights commit: `LLM(model=HUB_ID, revision="<full commit hash>")`.